In [1]:
from ultralytics import YOLO

In [56]:
det = YOLO("checkpoints/yolo11n.pt")
pose = YOLO("yolo11n-pose.pt")

100%|██████████| 5.97M/5.97M [00:00<00:00, 24.2MB/s]


In [57]:
pose.export(format = 'onnx', simplify=True)

Ultralytics 8.3.169  Python-3.12.6 torch-2.7.1+cu118 CPU (Intel Core(TM) i7-9750H 2.60GHz)
YOLO11n-pose summary (fused): 109 layers, 2,866,468 parameters, 0 gradients, 7.4 GFLOPs

PyTorch: starting from 'yolo11n-pose.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 56, 8400) (6.0 MB)
requirements: Ultralytics requirement ['onnxruntime-gpu'] not found, attempting AutoUpdate...
   --------------------------------------- 214.9/214.9 MB 38.8 MB/s eta 0:00:00

requirements: AutoUpdate success  16.0s
WARNING requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.17.0 opset 19...
ONNX: slimming with onnxslim 0.1.61...
ONNX: export success  19.4s, saved as 'yolo11n-pose.onnx' (11.2 MB)

Export complete (20.1s)
Results saved to E:\synclabs\edgeface
Predict:         yolo predict task=pose model=yolo11n-pose.onnx imgsz=640  
Validate:        yolo val task=pose model=yolo11n-pose.onnx imgsz=640 data=/ultralytics/ultralytics/c

'yolo11n-pose.onnx'

In [3]:
det_model = det.model
pose_model = pose.model

In [50]:
det._forward_hooks

OrderedDict()

In [47]:
det.__dict__

{'_parameters': {},
 '_buffers': {},
 '_non_persistent_buffers_set': set(),
 '_backward_pre_hooks': OrderedDict(),
 '_backward_hooks': OrderedDict(),
 '_is_full_backward_hook': None,
 '_forward_hooks': OrderedDict(),
 '_forward_hooks_with_kwargs': OrderedDict(),
 '_forward_hooks_always_called': OrderedDict(),
 '_forward_pre_hooks': OrderedDict(),
 '_forward_pre_hooks_with_kwargs': OrderedDict(),
 '_state_dict_hooks': OrderedDict(),
 '_state_dict_pre_hooks': OrderedDict(),
 '_load_state_dict_pre_hooks': OrderedDict(),
 '_load_state_dict_post_hooks': OrderedDict(),
 '_modules': {'model': DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        

In [4]:
print(det_model)

DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C3k2(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
   

In [11]:
import torch
import torch.nn as nn

def print_model_flow(model, input_size=(1, 3, 224, 224)):
    """
    Prints the flow of layers in a PyTorch model, showing input-output connections.
    
    Args:
        model (nn.Module): The PyTorch model whose flow is to be printed.
        input_size (tuple): The shape of the dummy input tensor to run the model.
    """
    def forward_hook(module, input, output):
        """
        Hook function to print the layer's input and output shapes.
        """
        input_shape = [str(x.shape) for x in input]
        output_shape = str(output.shape)
        print(f'{module.__class__.__name__.ljust(20)} | Input shape: {", ".join(input_shape):<40} | Output shape: {output_shape}')
    
    # Register forward hooks for each layer in the model
    hooks = []
    for layer in model.children():
        hook = layer.register_forward_hook(forward_hook)
        hooks.append(hook)
    
    # Create dummy input with the specified shape
    dummy_input = torch.randn(*input_size)
    
    print(f"{'Layer Name'.ljust(20)} | {'Input Shape':<40} | Output Shape")
    print("=" * 80)

    try:
        # Run a forward pass through the model to trigger the hooks
        model(dummy_input)  
    except Exception as e:
        print(f"Error during forward pass: {e}")
    
    # Remove hooks after use
    for hook in hooks:
        hook.remove()

# Example usage
# Assuming `model` is your PyTorch model
# print_model_flow(model)


In [13]:
import torch
import torch.nn as nn
from collections import defaultdict

def analyze_model_flow(model, input_size=(1, 3, 224, 224)):
    layer_info = {}
    connections = defaultdict(list)  # layer_id -> list of next layer_ids
    reverse_connections = defaultdict(list)  # layer_id -> list of previous layer_ids
    module_ids = {id(m): name for name, m in model.named_modules()}

    def hook_fn(module, inputs, outputs):
        layer_id = id(module)
        layer_name = module_ids[layer_id]
        
        # Store input/output shapes
        in_shapes = [tuple(x.shape) for x in inputs]
        if isinstance(outputs, (list, tuple)):
            out_shapes = [tuple(o.shape) for o in outputs]
        else:
            out_shapes = [tuple(outputs.shape)]
        
        layer_info[layer_id] = {
            "name": layer_name,
            "type": module.__class__.__name__,
            "input_shapes": in_shapes,
            "output_shapes": out_shapes,
            "num_inputs": len(in_shapes)
        }
        
        # Track connections
        for t in inputs:
            if hasattr(t, "_layer_id"):
                reverse_connections[layer_id].append(t._layer_id)
                connections[t._layer_id].append(layer_id)

        # Tag outputs so downstream layers know their source
        if isinstance(outputs, (list, tuple)):
            for o in outputs:
                if torch.is_tensor(o):
                    o._layer_id = layer_id
        else:
            outputs._layer_id = layer_id

    # Register hooks for all modules
    hooks = []
    for name, module in model.named_modules():
        hooks.append(module.register_forward_hook(hook_fn))

    # Forward pass with dummy input
    dummy_input = torch.randn(*input_size)
    model(dummy_input)

    # Remove hooks
    for h in hooks:
        h.remove()

    #


In [19]:
analyze_model_flow(pose_model.model)

TypeError: cat() received an invalid combination of arguments - got (Tensor, int), but expected one of:
 * (tuple of Tensors tensors, int dim = 0, *, Tensor out = None)
 * (tuple of Tensors tensors, name dim, *, Tensor out = None)


In [34]:
pose_model.modules

<bound method Module.modules of PoseModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C3k2(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiL

In [38]:
print(type(pose_model))        # <class 'ultralytics.nn.tasks.PoseModel' or similar>
print(type(pose_model.model))  # <class 'torch.nn.modules.container.Sequential'>


<class 'ultralytics.nn.tasks.PoseModel'>
<class 'torch.nn.modules.container.Sequential'>


In [40]:
import torch
import torch.nn as nn
from torchinfo import summary

In [42]:
summary(pose_model, input_size=(1, 3, 640,640))

RuntimeError: Failed to run torchinfo. See above stack traces for more details. Executed layers up to: [Conv: 2, Conv2d: 3, BatchNorm2d: 3, SiLU: 6, Conv: 2, Conv2d: 3, BatchNorm2d: 3, SiLU: 6, C3k2: 2, Conv: 3, Conv2d: 4, BatchNorm2d: 4, SiLU: 6, Bottleneck: 4, Conv: 5, Conv2d: 6, BatchNorm2d: 6, SiLU: 6, Conv: 5, Conv2d: 6, BatchNorm2d: 6, SiLU: 6, Conv: 3, Conv2d: 4, BatchNorm2d: 4, SiLU: 6, Conv: 2, Conv2d: 3, BatchNorm2d: 3, SiLU: 6, C3k2: 2, Conv: 3, Conv2d: 4, BatchNorm2d: 4, SiLU: 6, Bottleneck: 4, Conv: 5, Conv2d: 6, BatchNorm2d: 6, SiLU: 6, Conv: 5, Conv2d: 6, BatchNorm2d: 6, SiLU: 6, Conv: 3, Conv2d: 4, BatchNorm2d: 4, SiLU: 6, Conv: 2, Conv2d: 3, BatchNorm2d: 3, SiLU: 6, C3k2: 2, Conv: 3, Conv2d: 4, BatchNorm2d: 4, SiLU: 6, C3k: 4, Conv: 5, Conv2d: 6, BatchNorm2d: 6, SiLU: 6, Sequential: 5, Bottleneck: 6, Conv: 7, Conv2d: 8, BatchNorm2d: 8, SiLU: 6, Conv: 7, Conv2d: 8, BatchNorm2d: 8, SiLU: 6, Bottleneck: 6, Conv: 7, Conv2d: 8, BatchNorm2d: 8, SiLU: 6, Conv: 7, Conv2d: 8, B

In [25]:
print(pose_model.model)

Sequential(
  (0): Conv(
    (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
    (act): SiLU(inplace=True)
  )
  (1): Conv(
    (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
    (act): SiLU(inplace=True)
  )
  (2): C3k2(
    (cv1): Conv(
      (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (cv2): Conv(
      (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (m): ModuleList(
      (0): Bottleneck(
        (cv1): Conv(
       